In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches


# Model agnostic 
from typing import Optional, List, Callable, Dict, Any, List
from pathlib import Path
from src.utils import Helm, QLatticeWrapper
from src.dfm_src import DFT

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
# Get the directory this file lives in
nb_dir = Path.cwd() # notebook directory
project_root = nb_dir.parents[0] # project directory
data_path = project_root / "datasets" / "processed_well_data.csv"

includ_cols = ['Dia', 'Dev(deg)','Area (m2)', 'z','GasDens','LiquidDens', 'g (m/s2)', 'P/T','friction_factor', 'critical_film_thickness']
D = Helm(path=data_path, includ_cols=includ_cols, test_size=0.20, scale=False) # cannot scale input units 

In [3]:
# define xgboost pipeline
def dft(
        hparams: Dict[str, Any]
):
    dft = DFT(
        **hparams,
    )

    return dft

hparam_grid = {
            "dev_tol":  [1e-5], # [1e-5, 1e-4, 1e-2],
            "feature_tol": [1.0], #[0.5, 1.0, 2.0],
        }
hparam_grid["interval"] = [0]

# train model and optimize hyperparameters via grid search 
trained_model = D.evolv_model(build_model=dft, hparam_grid=hparam_grid, k_folds=5)

Training model and optimizing hyperparameters via k-fold CV...
Retraining optimized model on full training set...

Best CV Classification Accuracy: 
>>> 0.6870 ± 0.0545

Best CV Per-Class Accuracy:
>>> Unloaded (-1): 0.6705, Near L.U. (0): 0.0000, Loaded (1): 0.7534

Best Hyperparameters:
>>> {'dev_tol': 1e-05, 'feature_tol': 1.0, 'interval': 0}

Training Classification Accuracy:
>>> 0.9398

Per-Class Training Accuracy:
>>> Unloaded (-1): 0.9432, Near L.U. (0): 0.0000, Loaded (1): 1.0000

Training Regression Metrics:
>>> RMSE=16720.9976, MAE=6743.3323, R2=0.9823

Test Regression Metrics:
>>> RMSE=390213.1022, MAE=177526.3205, R2=-9.2691

Test Classification Accuracy:
>>> 0.7143

Per-Class Test Accuracy:
>>> Unloaded (-1): 0.6957, Near L.U. (0): 0.0000, Loaded (1): 0.7778
